In [33]:
%%writefile option_pricer.cu
/*
 * option_pricer.cu
 * CUDA American Option Pricer — CRR Binomial Tree
 *
 * Compile:  nvcc -O2 -o option_pricer option_pricer.cu -lm
 * Run:      ./option_pricer
 */

#include <cuda_runtime.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>


#define N_STEPS      100
#define MAX_NODES    (N_STEPS + 1)
#define IV_ITERS     40
#define IV_LOW       0.0001f
#define IV_HIGH      4.0f
#define BENCH_N      100000

typedef struct {
    float S0;
    float K;
    float T;
    float r;
    float sigma;
    int   is_call;
} OptionParams;

typedef struct {
    float S0;
    float K;
    float T;
    float r;
    float market_price;
    int   is_call;
} IVParams;

__device__ float price_tree_device(float S0, float K, float T, float r,
                                   float sigma, int is_call)
{
    if (T <= 0.0f)
        return is_call ? fmaxf(S0 - K, 0.0f) : fmaxf(K - S0, 0.0f);

    float dt = T / N_STEPS;
    float u  = expf(sigma * sqrtf(dt));
    float d  = 1.0f / u;
    float df = expf(-r * dt);
    float p  = (expf(r * dt) - d) / (u - d);

    float vals[MAX_NODES];

    for (int j = 0; j <= N_STEPS; ++j) {
        float st = S0 * powf(u, (float)(N_STEPS - j)) * powf(d, (float)j);
        vals[j] = is_call ? fmaxf(st - K, 0.0f) : fmaxf(K - st, 0.0f);
    }

    for (int level = N_STEPS - 1; level >= 0; --level) {
        for (int j = 0; j <= level; ++j) {
            float cont = (p * vals[j] + (1.0f - p) * vals[j + 1]) * df;
            float st   = S0 * powf(u, (float)(level - j)) * powf(d, (float)j);
            float exer = is_call ? fmaxf(st - K, 0.0f) : fmaxf(K - st, 0.0f);
            vals[j]    = fmaxf(cont, exer);
        }
    }
    return vals[0];
}

__global__ void price_options_kernel(
    const OptionParams* __restrict__ opts,
    float*             __restrict__ results,
    int                             n_options)
{
    int opt_idx = blockIdx.x;
    if (opt_idx >= n_options) return;

    __shared__ float vals[MAX_NODES];

    OptionParams o   = opts[opt_idx];
    int          tid = threadIdx.x;

    if (o.T <= 0.0f) {
        if (tid == 0)
            results[opt_idx] = o.is_call ? fmaxf(o.S0 - o.K, 0.0f) : fmaxf(o.K - o.S0, 0.0f);
        return;
    }

    float dt = o.T / N_STEPS;
    float u  = expf(o.sigma * sqrtf(dt));
    float d  = 1.0f / u;
    float df = expf(-o.r * dt);
    float p  = (expf(o.r * dt) - d) / (u - d);

    for (int j = tid; j <= N_STEPS; j += blockDim.x) {
        float st = o.S0 * powf(u, (float)(N_STEPS - j)) * powf(d, (float)j);
        vals[j] = o.is_call ? fmaxf(st - o.K, 0.0f) : fmaxf(o.K - st, 0.0f);
    }
    __syncthreads();

    for (int level = N_STEPS - 1; level >= 0; --level) {
        for (int j = tid; j <= level; j += blockDim.x) {
            float cont = (p * vals[j] + (1.0f - p) * vals[j + 1]) * df;
            float st   = o.S0 * powf(u, (float)(level - j)) * powf(d, (float)j);
            float exer = o.is_call ? fmaxf(st - o.K, 0.0f) : fmaxf(o.K - st, 0.0f);
            vals[j] = fmaxf(cont, exer);
        }
        __syncthreads();
    }

    if (tid == 0)
        results[opt_idx] = vals[0];

}

__global__ void price_options_kernel_opt(
    const OptionParams* __restrict__ opts,
    float*              __restrict__ results,
    int                              n_options)
{
    int opt_idx = blockIdx.x;
    if (opt_idx >= n_options) return;

    extern __shared__ float shared[];
    OptionParams o   = opts[opt_idx];
    int          tid = threadIdx.x;
    int          N   = N_STEPS;
    int          bsz = blockDim.x;

    float* vals = shared;
    float* spot = shared + (N + 1);

    if (o.T <= 0.0f) {
        if (tid == 0)
            results[opt_idx] = o.is_call ? fmaxf(o.S0-o.K,0.0f) : fmaxf(o.K-o.S0,0.0f);
        return;
    }

    float dt = o.T / N;
    float u  = expf(o.sigma * sqrtf(dt));
    float d  = 1.0f / u;
    float df = expf(-o.r * dt);
    float p  = (expf(o.r * dt) - d) / (u - d);

    /* All threads fill terminal in parallel using powf — same as original */
    for (int j = tid; j <= N; j += bsz) {
        float S = o.S0 * powf(u, (float)(N - j)) * powf(d, (float)j);
        spot[j] = S;
        vals[j] = o.is_call ? fmaxf(S-o.K,0.0f) : fmaxf(o.K-S,0.0f);
    }
    __syncthreads();

    /* Backward induction — spot[j] *= d each level, no powf */
    for (int level = N-1; level >= 0; --level) {
        for (int j = tid; j <= level; j += bsz) {
            spot[j] *= d;
            float cont = (p*vals[j] + (1.0f-p)*vals[j+1]) * df;
            float exer = o.is_call ? fmaxf(spot[j]-o.K,0.0f) : fmaxf(o.K-spot[j],0.0f);
            vals[j] = fmaxf(cont, exer);
        }
        __syncthreads();
    }

    if (tid == 0) results[opt_idx] = vals[0];
}

__global__ void solve_iv_kernel(
    const IVParams* __restrict__ iv_opts,
    float*         __restrict__ iv_results,
    int                         n_options)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n_options) return;

    IVParams iv = iv_opts[idx];
    float lo = IV_LOW, hi = IV_HIGH;

    for (int i = 0; i < IV_ITERS; ++i) {
        float mid   = (lo + hi) * 0.5f;
        float price = price_tree_device(iv.S0, iv.K, iv.T, iv.r, mid, iv.is_call);
        if (price < iv.market_price) lo = mid;
        else                         hi = mid;
    }

    iv_results[idx] = (lo + hi) * 0.5f;
}

void price_options_gpu(OptionParams* h_opts, float* h_results, int n)
{
    OptionParams* d_opts;
    float*        d_results;
    cudaMalloc(&d_opts,    n * sizeof(OptionParams));
    cudaMalloc(&d_results, n * sizeof(float));
    cudaMemcpy(d_opts, h_opts, n * sizeof(OptionParams), cudaMemcpyHostToDevice);
    price_options_kernel<<<n, 128>>>(d_opts, d_results, n);
    cudaDeviceSynchronize();
    cudaMemcpy(h_results, d_results, n * sizeof(float), cudaMemcpyDeviceToHost);
    cudaFree(d_opts);
    cudaFree(d_results);
}

void price_options_gpu_opt(OptionParams* h_opts, float* h_results, int n)
{
    int threads = 32;
    while (threads < N_STEPS && threads < 1024) threads *= 2;
    size_t shmem = 2 * (N_STEPS + 1) * sizeof(float);
    OptionParams* d_opts;
    float*        d_results;
    cudaMalloc(&d_opts,    n * sizeof(OptionParams));
    cudaMalloc(&d_results, n * sizeof(float));
    cudaMemcpy(d_opts, h_opts, n * sizeof(OptionParams), cudaMemcpyHostToDevice);
    price_options_kernel_opt<<<n, threads, shmem>>>(d_opts, d_results, n);
    cudaDeviceSynchronize();
    cudaMemcpy(h_results, d_results, n * sizeof(float), cudaMemcpyDeviceToHost);
    cudaFree(d_opts);
    cudaFree(d_results);
}

void solve_iv_gpu(IVParams* h_iv, float* h_results, int n)
{
    IVParams* d_iv;
    float*    d_results;
    cudaMalloc(&d_iv,      n * sizeof(IVParams));
    cudaMalloc(&d_results, n * sizeof(float));
    cudaMemcpy(d_iv, h_iv, n * sizeof(IVParams), cudaMemcpyHostToDevice);
    int threads = 256;
    int blocks  = (n + threads - 1) / threads;
    solve_iv_kernel<<<blocks, threads>>>(d_iv, d_results, n);
    cudaDeviceSynchronize();
    cudaMemcpy(h_results, d_results, n * sizeof(float), cudaMemcpyDeviceToHost);
    cudaFree(d_iv);
    cudaFree(d_results);
}

float price_option_cpu(OptionParams o)
{
    if (o.T <= 0.0f)
        return o.is_call ? fmaxf(o.S0 - o.K, 0.0f) : fmaxf(o.K - o.S0, 0.0f);

    float dt = o.T / N_STEPS;
    float u  = expf(o.sigma * sqrtf(dt));
    float d  = 1.0f / u;
    float df = expf(-o.r * dt);
    float p  = (expf(o.r * dt) - d) / (u - d);

    static float vals[MAX_NODES];
    for (int j = 0; j <= N_STEPS; ++j) {
        float st = o.S0 * powf(u, (float)(N_STEPS - j)) * powf(d, (float)j);
        vals[j] = o.is_call ? fmaxf(st - o.K, 0.0f) : fmaxf(o.K - st, 0.0f);
    }
    for (int level = N_STEPS - 1; level >= 0; --level) {
        for (int j = 0; j <= level; ++j) {
            float cont = (p * vals[j] + (1.0f - p) * vals[j + 1]) * df;
            float st   = o.S0 * powf(u, (float)(level - j)) * powf(d, (float)j);
            float exer = o.is_call ? fmaxf(st - o.K, 0.0f) : fmaxf(o.K - st, 0.0f);
            vals[j]    = fmaxf(cont, exer);
        }
    }
    return vals[0];
}

float solve_iv_cpu(IVParams iv)
{
    float lo = IV_LOW, hi = IV_HIGH;
    for (int i = 0; i < IV_ITERS; ++i) {
        float mid   = (lo + hi) * 0.5f;
        OptionParams o = {iv.S0, iv.K, iv.T, iv.r, mid, iv.is_call};
        float price = price_option_cpu(o);
        if (price < iv.market_price) lo = mid; else hi = mid;
    }
    return (lo + hi) * 0.5f;
}

static double now_ms(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec * 1e3 + ts.tv_nsec * 1e-6;
}

static float cuda_time_ms(cudaEvent_t a, cudaEvent_t b)
{
    float ms;
    cudaEventElapsedTime(&ms, a, b);
    return ms;
}

int main(void)
{
    printf("=== CUDA American Option Pricer v2 ===\n\n");

    printf("== Section 1: IV Solver Correctness ==\n");

    IVParams known_iv = {
        .S0           = 575.0f,
        .K            = 575.0f,
        .T            = 30.0f / 365.0f,
        .r            = 0.045f,
        .market_price = 12.88f,
        .is_call      = 1
    };

    float gpu_iv, cpu_iv;
    solve_iv_gpu(&known_iv, &gpu_iv, 1);
    cpu_iv = solve_iv_cpu(known_iv);

    printf("  Market price used:  $%.2f\n", known_iv.market_price);
    printf("  GPU implied vol:    %.4f%%\n", gpu_iv * 100.0f);
    printf("  CPU implied vol:    %.4f%%\n", cpu_iv * 100.0f);
    printf("  IV diff:            %.6f%%\n", fabsf(gpu_iv - cpu_iv) * 100.0f);

    OptionParams roundtrip = {
        known_iv.S0, known_iv.K, known_iv.T,
        known_iv.r, gpu_iv, known_iv.is_call
    };
    float reprice;
    price_options_gpu(&roundtrip, &reprice, 1);
    printf("  Round-trip price:   $%.4f  (target $%.2f, diff $%.4f)\n\n",
           reprice, known_iv.market_price, fabsf(reprice - known_iv.market_price));

    printf("  Strike sweep IV:\n");
    printf("  %-10s %-14s %-14s %-12s\n", "Strike", "GPU IV", "CPU IV", "|Diff|");
    printf("  %s\n", "--------------------------------------------------");

    float mkt_prices[] = {38.32f, 29.77f, 25.83f, 22.15f, 18.75f,
                          15.66f, 12.88f, 10.49f,  6.66f,  3.94f};
    float strikes[]    = {540,550,555,560,565,570,575,580,590,600};
    int   n_iv         = 10;

    IVParams* h_iv_in  = (IVParams*)malloc(n_iv * sizeof(IVParams));
    float*    h_iv_out = (float*)malloc(n_iv * sizeof(float));

    for (int i = 0; i < n_iv; ++i)
        h_iv_in[i] = (IVParams){575.0f, strikes[i], 30.0f/365.0f, 0.045f, mkt_prices[i], 1};

    solve_iv_gpu(h_iv_in, h_iv_out, n_iv);

    for (int i = 0; i < n_iv; ++i) {
        float c_iv = solve_iv_cpu(h_iv_in[i]);
        printf("  %-10.0f %-14.4f %-14.4f %-12.6f\n",
               strikes[i], h_iv_out[i]*100.0f, c_iv*100.0f, fabsf(h_iv_out[i]-c_iv)*100.0f);
    }

    free(h_iv_in);
    free(h_iv_out);

    printf("\n== Section 2: Benchmarks (%d options) ==\n", BENCH_N);

    OptionParams* bench_opts = (OptionParams*)malloc(BENCH_N * sizeof(OptionParams));
    IVParams*     bench_iv   = (IVParams*)malloc(BENCH_N * sizeof(IVParams));
    float*        bench_out  = (float*)malloc(BENCH_N * sizeof(float));

    for (int i = 0; i < BENCH_N; ++i) {
        float K     = 500.0f + (i % 200);
        float T     = (float)((i % 90) + 1) / 365.0f;
        float sigma = 0.10f + (i % 30) * 0.01f;
        bench_opts[i] = (OptionParams){575.0f, K, T, 0.045f, sigma, (i%2==0)};
        bench_iv[i]   = (IVParams){575.0f, K, T, 0.045f, 5.0f + (i % 60) * 0.5f, (i%2==0)};
    }

    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0);
    cudaEventCreate(&ev1);

    price_options_gpu(bench_opts, bench_out, BENCH_N);
    cudaEventRecord(ev0);
    price_options_gpu(bench_opts, bench_out, BENCH_N);
    cudaEventRecord(ev1); cudaEventSynchronize(ev1);
    float gpu_price_ms = cuda_time_ms(ev0, ev1);

    price_options_gpu_opt(bench_opts, bench_out, BENCH_N);
    cudaEventRecord(ev0);
    price_options_gpu_opt(bench_opts, bench_out, BENCH_N);
    cudaEventRecord(ev1); cudaEventSynchronize(ev1);
    float gpu_price_opt_ms = cuda_time_ms(ev0, ev1);

    double cpu_start = now_ms();
    for (int i = 0; i < BENCH_N; ++i)
        bench_out[i] = price_option_cpu(bench_opts[i]);
    double cpu_price_ms = now_ms() - cpu_start;

    printf("\n  -- Option Pricing --\n");
    printf("  GPU original:     %8.2f ms   (%8.0f opts/sec)  speedup: %.0fx\n",
           gpu_price_ms, BENCH_N/(gpu_price_ms/1000.0), cpu_price_ms/gpu_price_ms);
    printf("  GPU optimized:    %8.2f ms   (%8.0f opts/sec)  speedup: %.0fx\n",
           gpu_price_opt_ms, BENCH_N/(gpu_price_opt_ms/1000.0), cpu_price_ms/gpu_price_opt_ms);
    printf("  CPU time:         %8.2f ms   (%8.0f opts/sec)\n",
           cpu_price_ms, BENCH_N/(cpu_price_ms/1000.0));
    printf("  OPT vs ORIG:      %.2fx faster\n", gpu_price_ms/gpu_price_opt_ms);

    solve_iv_gpu(bench_iv, bench_out, BENCH_N);
    cudaEventRecord(ev0);
    solve_iv_gpu(bench_iv, bench_out, BENCH_N);
    cudaEventRecord(ev1); cudaEventSynchronize(ev1);
    float gpu_iv_ms = cuda_time_ms(ev0, ev1);

    cpu_start = now_ms();
    int cpu_iv_sample = 500;
    for (int i = 0; i < cpu_iv_sample; ++i)
        bench_out[i] = solve_iv_cpu(bench_iv[i]);
    double cpu_iv_ms = (now_ms() - cpu_start) / cpu_iv_sample * BENCH_N;

    printf("\n  -- IV Solving --\n");
    printf("  GPU kernel time:  %8.2f ms   (%8.0f solves/sec)\n",
           gpu_iv_ms, BENCH_N / (gpu_iv_ms / 1000.0));
    printf("  CPU time (est):   %8.2f ms   (%8.0f solves/sec)   [from %d samples]\n",
           cpu_iv_ms, BENCH_N / (cpu_iv_ms / 1000.0), cpu_iv_sample);
    printf("  Speedup:          %8.2fx\n", cpu_iv_ms / gpu_iv_ms);

    printf("\n  -- End-to-end wall time (kernel + memcpy) --\n");
    double e2e_gpu_start = now_ms();
    price_options_gpu(bench_opts, bench_out, BENCH_N);
    double e2e_gpu_ms = now_ms() - e2e_gpu_start;
    printf("  GPU (kernel+copy): %.2f ms\n", e2e_gpu_ms);
    printf("  CPU sequential:    %.2f ms\n", cpu_price_ms);
    printf("  Wall speedup:      %.2fx\n\n", cpu_price_ms / e2e_gpu_ms);

    /* -- Correctness check -------------------------------------------- */
    float* check_orig = (float*)malloc(1000 * sizeof(float));
    float* check_opt  = (float*)malloc(1000 * sizeof(float));
    price_options_gpu(bench_opts, check_orig, 1000);
    price_options_gpu_opt(bench_opts, check_opt, 1000);

    printf("  -- Correctness debug (first 3 options) --\n");
    for (int i = 0; i < 3; i++)
        printf("  opt[%d]=%.4f  orig[%d]=%.4f  cpu[%d]=%.4f\n",
               i, check_opt[i], i, check_orig[i], i, price_option_cpu(bench_opts[i]));

    float max_diff = 0.0f;
    for (int i = 0; i < 1000; i++) {
        float diff = fabsf(check_orig[i] - check_opt[i]);
        if (diff > max_diff) max_diff = diff;
    }
    printf("\n  -- Correctness (orig vs opt, 1000 options) --\n");
    printf("  Max diff: $%.6f  %s\n\n", max_diff, max_diff < 0.01f ? "PASS" : "FAIL");

    free(check_orig);
    free(check_opt);
    cudaEventDestroy(ev0);
    cudaEventDestroy(ev1);
    free(bench_opts);
    free(bench_iv);
    free(bench_out);

    return 0;
}

Overwriting option_pricer.cu


In [28]:
!nvidia-smi

Wed Apr 15 21:07:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [34]:
!nvcc -O3 -o option_pricer option_pricer.cu -lm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [35]:
!./option_pricer

=== CUDA American Option Pricer v2 ===

== Section 1: IV Solver Correctness ==
  Market price used:  $12.88
  GPU implied vol:    18.0064%
  CPU implied vol:    18.0069%
  IV diff:            0.000480%
  Round-trip price:   $12.8800  (target $12.88, diff $0.0000)

  Strike sweep IV:
  Strike     GPU IV         CPU IV         |Diff|      
  --------------------------------------------------
  540        18.0018        18.0028        0.000957    
  550        17.9903        17.9908        0.000489    
  555        17.9981        17.9986        0.000566    
  560        18.0035        18.0036        0.000054    
  565        17.9998        18.0003        0.000492    
  570        18.0073        18.0077        0.000444    
  575        18.0064        18.0069        0.000480    
  580        17.9969        17.9970        0.000066    
  590        18.0044        18.0044        0.000015    
  600        18.0076        18.0077        0.000155    

== Section 2: Benchmarks (100000 options) ==

